# 📓 Qdislib Hello World — CUDA-Q backend

This notebook mirrors the serial *Hello World* example, but builds the circuit as a
[CUDA-Q](https://nvidia.github.io/cuda-quantum/) kernel and runs the circuit-cutting
workflow with `software="cudaq"`. The same Qdislib API works for Qibo, Qiskit and CUDA-Q.


### 🔧 1. Install Qdislib with the CUDA-Q backend

`!pip install "qdislib[cudaq]"`

CUDA-Q runs on CPU (`qpp-cpu`) out of the box and uses the GPU automatically when available.


### 🧠 2. Importing Required Modules


In [ ]:
import numpy as np
import cudaq

import Qdislib.api as qd


### ⚙️ 3. Define the Main Circuit as a CUDA-Q kernel

Qdislib introspects the kernel source, so define it with the usual CUDA-Q builder syntax.


In [ ]:
@cudaq.kernel
def entire_circuit():
    q = cudaq.qvector(5)
    h(q[0])
    z.ctrl(q[0], q[1])
    z.ctrl(q[1], q[2])
    ry(0.6, q[2])
    z.ctrl(q[2], q[3])
    z.ctrl(q[3], q[4])
    rx(0.4, q[1])

print(cudaq.draw(entire_circuit))


### 🔪 4. Find a cut

`find_cut` is backend-agnostic and works directly on a CUDA-Q kernel.


In [ ]:
cut = qd.find_cut(entire_circuit, gate_cut=True, wire_cut=False)
print(cut)


['CZ_3']


### ✂️ 5. Gate Cutting — recommended workflow

The recommended pattern is three steps: **generate** the subcircuits, **evaluate** each
one in your own loop (here with the built-in `qd.expectation_value`, but this is where you
would dispatch to a simulator, a GPU, or real hardware), then **reconstruct**.


In [ ]:
# 1. Cut the circuit into independently executable subcircuits.
subcircuits = qd.gate_cutting_subcircuits(entire_circuit, cut, software="cudaq")

# 2. Evaluate each subcircuit (this loop is yours to control).
values = [qd.expectation_value(s, shots=8192) for s in subcircuits]

# 3. Recombine into the original circuit's expectation value.
reconstruction = subcircuits.reconstruct(values)
print(reconstruction)


0.0171


### 🔌 6. Wire Cutting

Wire cutting follows the exact same three-step pattern.


In [ ]:
wire_cut = qd.find_cut(entire_circuit, gate_cut=False, wire_cut=True)
print(wire_cut)

subcircuits = qd.wire_cutting_subcircuits(entire_circuit, wire_cut, software="cudaq")
values = [qd.expectation_value(s, shots=8192) for s in subcircuits]
print(subcircuits.reconstruct(values))


[('CZ_3', 'RY_4')]


-0.0141


### 📊 7. Exact Expectation Value (reference)

For comparison, the exact value of the uncut circuit. The reconstructed values above
match it within shot noise.


In [ ]:
from qibo import models, gates

qc = models.Circuit(5)
qc.add(gates.H(0)); qc.add(gates.CZ(0, 1)); qc.add(gates.CZ(1, 2))
qc.add(gates.RY(2, 0.6)); qc.add(gates.CZ(2, 3)); qc.add(gates.CZ(3, 4)); qc.add(gates.RX(1, 0.4))

print(qd.analytical_solution(qc, "Z" * qc.nqubits))


0.0


### ✅ Summary

The same generate → evaluate → reconstruct workflow drives Qibo, Qiskit and CUDA-Q —
only the `software=` argument changes. See the other notebooks for the Qibo/Qiskit and
IBM Quantum QPU variants.
